In [1]:
import re

FILE = '/workspaces/scripture-journey/data/prophecies.ts'
with open(FILE, 'r') as f:
    content = f.read()

# IDs to remove
REMOVE_IDS = {101,102,103,106,113,131,132,135,138,141,146,148,154,158,164,172,173,176,177}

# Build ordered list of old IDs that survive (plus 212 for new entry)
kept_old_ids = [i for i in range(1, 212) if i not in REMOVE_IDS]
kept_old_ids.append(212)  # new Hosea 6:2 entry
assert len(kept_old_ids) == 193, f"Expected 193, got {len(kept_old_ids)}"

# Build old-to-new mapping
old_to_new = {old_id: new_id for new_id, old_id in enumerate(kept_old_ids, 1)}
print(f"Mapping built: {len(old_to_new)} entries")
print(f"Sample: old 104 -> new {old_to_new[104]}, old 211 -> new {old_to_new[211]}, new 212 -> {old_to_new[212]}")

# ===== STEP 1: Remove 19 push blocks =====
def find_push_block(text, old_id):
    marker = f'prophecies.push(makeLesson({old_id}, '
    idx = text.find(marker)
    if idx == -1:
        marker = f'prophecies.push(makeLesson({old_id},"'
        idx = text.find(marker)
    if idx == -1:
        raise ValueError(f"Could not find push block for ID {old_id}")

    line_start = text.rfind('\n', 0, idx) + 1

    if line_start > 1:
        prev_end = line_start - 1
        prev_start = text.rfind('\n', 0, prev_end) + 1
        prev_line = text[prev_start:prev_end].strip()
        if prev_line.startswith('//'):
            line_start = prev_start

    depth = 0
    in_str = False
    esc = False
    str_ch = ''
    end = -1
    for i in range(idx, len(text)):
        c = text[i]
        if esc:
            esc = False
            continue
        if c == '\\' and in_str:
            esc = True
            continue
        if in_str:
            if c == str_ch:
                in_str = False
            continue
        if c in ('"', "'", '`'):
            in_str = True
            str_ch = c
            continue
        if c == '(':
            depth += 1
        if c == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end == -1:
        raise ValueError(f"Could not find end of push block for ID {old_id}")

    while end < len(text) and text[end] == ';':
        end += 1
    while end < len(text) and text[end] == '\n':
        end += 1

    return line_start, end

for old_id in sorted(REMOVE_IDS, reverse=True):
    start, end = find_push_block(content, old_id)
    content = content[:start] + content[end:]
    print(f"Removed block for old ID {old_id}")

print("All 19 blocks removed")

Mapping built: 193 entries
Sample: old 104 -> new 101, old 211 -> new 192, new 212 -> 193
Removed block for old ID 177
Removed block for old ID 176
Removed block for old ID 173
Removed block for old ID 172
Removed block for old ID 164
Removed block for old ID 158
Removed block for old ID 154
Removed block for old ID 148
Removed block for old ID 146
Removed block for old ID 141
Removed block for old ID 138
Removed block for old ID 135
Removed block for old ID 132
Removed block for old ID 131
Removed block for old ID 113
Removed block for old ID 106
Removed block for old ID 103
Removed block for old ID 102
Removed block for old ID 101
All 19 blocks removed


In [2]:
# ===== STEP 2: Add new Hosea 6:2 entry before the for loop =====

new_entry = 'prophecies.push(makeLesson(212, "raised-on-third-day", "Raised on the Third Day", "Resurrection",\n  "Hosea 6:2", "Luke 24:46",\n  "After two days he will revive us; on the third day he will restore us, that we may live in his presence.",\n  "He told them, \\"This is what is written: The Messiah will suffer and rise from the dead on the third day,\\"",\n  "Hosea\\u2019s prophecy of revival after two days and restoration on the third established a prophetic pattern that Jesus directly cited. When He explained the Scriptures after His resurrection, He pointed to this very pattern \\u2014 that it was written the Messiah would rise on the third day. This is one of the most direct prophetic anticipations of the resurrection in the entire Old Testament.",\n  undefined,\n  { ...payne(78, "Hos 6:2", "Revival after two days and raising on the third day; Payne treats this as a prophetic pattern directly fulfilled in Christ\\u2019s resurrection \\u2014 listed as #78 in his 191 messianic prophecies"), ...edersheim("Hos. vi. 2 is Messianically applied in the Targum \\u2014 explicitly confirmed in Appendix IX of The Life and Times of Jesus the Messiah") }));\n\n'

for_loop_marker = 'for (const p of prophecies)'
for_idx = content.find(for_loop_marker)
if for_idx == -1:
    raise ValueError("Could not find for loop marker")
content = content[:for_idx] + new_entry + content[for_idx:]
print("Added new Hosea 6:2 entry")

# ===== STEP 3: Renumber makeLesson IDs (two-pass to avoid collisions) =====
# Pass 1: old -> temp placeholder
for old_id in sorted(old_to_new.keys(), reverse=True):
    new_id = old_to_new[old_id]
    content = content.replace(f'makeLesson({old_id},', f'makeLesson(__TEMP_{new_id}__,')

# Pass 2: temp -> new
content = re.sub(r'makeLesson\(__TEMP_(\d+)__,', lambda m: f'makeLesson({m.group(1)},', content)
print("Renumbered all makeLesson IDs")

# ===== STEP 4: Replace _prophecyTypeMap =====
old_type_map_match = re.search(r'const _prophecyTypeMap.*?= \{([\s\S]*?)\};', content)
old_type_content = old_type_map_match.group(1)
old_types = {}
for m in re.finditer(r'(\d+):\s*\'([^\']+)\'', old_type_content):
    old_types[int(m.group(1))] = m.group(2)

# All old types have been renumbered via the temp replacement
# But wait - the type map has its OWN set of key-value pairs that were NOT renamed
# The temp replacement only affected makeLesson() calls, not the type map keys
# So really the type map keys are still the OLD numbers
# Let me re-parse: actually, the type map is a Record<number, ProphecyType> 
# with hard-coded number keys. These are NOT affected by the makeLesson renaming.
# So I need to rebuild the type map from scratch.

# Re-read the type map from the ORIGINAL file to get authentic types per old ID
with open(FILE, 'r') as f:
    orig = f.read()

# Wait, no - the file was already modified by the removal step.
# But the type map itself wasn't changed by removals (it's separate from the push blocks).
# However, the makeLesson renaming DID affect it because the type map contains numbers
# that could have been caught by the global replace.
# Actually, let me check: the type map contains entries like `1: 'Direct Prophecy'`
# and the replace pattern was `makeLesson(OLD,` -> `makeLesson(__TEMP_NEW__,`
# Since the type map doesn't contain `makeLesson(`, the keys were NOT affected. Good.

# But we need to parse the type map that's currently in `content` (which still has old keys)
# and rebuild it with new keys.

# Actually, I realize there's a problem. Let me re-read the original file's type map.
# Since steps 1-3 changed the content variable, but the type map in content still has
# old-numbered keys (since we only renamed makeLesson calls, not Map keys).

# Let me parse the type map from content now
type_map_match = re.search(r'const _prophecyTypeMap.*?= \{([\s\S]*?)\};', content)
type_content = type_map_match.group(1)
current_types = {}
for m in re.finditer(r'(\d+):\s*\'([^\']+)\'', type_content):
    current_types[int(m.group(1))] = m.group(2)

print(f"Parsed {len(current_types)} type entries from current content")
print(f"Keys include: {sorted(current_types.keys())[:10]}...{sorted(current_types.keys())[-5:]}")

# Build new type map keyed by new IDs
new_types = {}
for old_id, new_id in old_to_new.items():
    if old_id == 212:
        new_types[new_id] = 'Prophetic Pattern'
    elif old_id in current_types:
        new_types[new_id] = current_types[old_id]

# Check if we have all 193 entries
for i in range(1, 194):
    if i not in new_types:
        # Use the default from makeLesson: 'Direct Prophecy'
        new_types[i] = 'Direct Prophecy'
        print(f"  Warning: No type for new ID {i}, defaulting to Direct Prophecy")

# Format the new type map (3 per line for 1-100, 2 per line after)
lines = []
entries = []
for new_id in range(1, 194):
    t = new_types[new_id]
    entries.append(f"{new_id}: '{t}'")
    max_per = 3 if new_id <= 100 else 2
    if len(entries) >= max_per or new_id == 193:
        lines.append('  ' + ', '.join(entries) + ',')
        entries = []
if entries:
    lines.append('  ' + ', '.join(entries) + ',')

new_type_block = f"const _prophecyTypeMap: Record<number, ProphecyType> = {{\n" + '\n'.join(lines) + "\n};"

old_block = type_map_match.group(0)
content = content.replace(old_block, new_type_block)
print(f"Rebuilt _prophecyTypeMap with {len(new_types)} entries")

# ===== STEP 5: Update _scholarshipMap =====
# Keys 1-98 don't change. Key 195->176, key 203->184, add key 193.

# Use precise patterns to target scholarship map keys
# The scholarship map is between `const _scholarshipMap...= {` and `};`
schol_match = re.search(r'(const _scholarshipMap.*?= \{)([\s\S]*?)(\n\};)', content)
schol_inner = schol_match.group(2)

# Replace key 195 with 176
schol_inner = re.sub(r'^(  )195:', r'\g<1>176:', schol_inner, count=1, flags=re.MULTILINE)
# Replace key 203 with 184
schol_inner = re.sub(r'^(  )203:', r'\g<1>184:', schol_inner, count=1, flags=re.MULTILINE)

# Add entry for 193 at the end
new_schol = '  193: { ...payne(78, "Hos 6:2", "Revival after two days and raising on the third day; Payne treats this as a prophetic pattern directly fulfilled in Christ\\u2019s resurrection \\u2014 listed as #78 in his 191 messianic prophecies"), ...edersheim("Hos. vi. 2 is Messianically applied in the Targum \\u2014 explicitly confirmed in Appendix IX of The Life and Times of Jesus the Messiah") },'
schol_inner = schol_inner.rstrip() + '\n' + new_schol + '\n'

content = content[:schol_match.start()] + schol_match.group(1) + schol_inner + schol_match.group(3) + content[schol_match.end():]
print("Updated _scholarshipMap (keys 195->176, 203->184, added 193)")

# ===== WRITE RESULT =====
with open(FILE, 'w') as f:
    f.write(content)
print("\nFile written successfully!")

Added new Hosea 6:2 entry
Renumbered all makeLesson IDs
Parsed 211 type entries from current content
Keys include: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...[207, 208, 209, 210, 211]
Rebuilt _prophecyTypeMap with 193 entries
Updated _scholarshipMap (keys 195->176, 203->184, added 193)

File written successfully!


In [3]:
# ===== VERIFICATION =====
with open(FILE, 'r') as f:
    final = f.read()

# Count makeLesson calls
make_lesson_calls = re.findall(r'makeLesson\((\d+),', final)
total_lessons = len(make_lesson_calls)
print(f"Total makeLesson calls: {total_lessons}")

# Check sequential numbering
ids = [int(m) for m in make_lesson_calls]
expected = list(range(1, 194))
if ids == expected:
    print("IDs are sequential 1-193 ✓")
else:
    missing = sorted(set(expected) - set(ids))
    extra = sorted(set(ids) - set(expected))
    dups = sorted([x for x in ids if ids.count(x) > 1])
    print(f"ID mismatch! Missing: {missing[:10]}, Extra: {extra[:10]}, Dups: {set(dups)}")
    for i, (actual, exp) in enumerate(zip(ids, expected)):
        if actual != exp:
            print(f"  First mismatch at position {i}: got {actual}, expected {exp}")
            break

# Check type map
type_map_match = re.search(r'const _prophecyTypeMap.*?= \{([\s\S]*?)\};', final)
type_keys = [int(m) for m in re.findall(r'(\d+):', type_map_match.group(1))]
print(f"_prophecyTypeMap keys: {len(type_keys)} (should be 193)")
expected_keys = list(range(1, 194))
if type_keys == expected_keys:
    print("Type map keys are sequential 1-193 ✓")
else:
    print(f"Type map key issues: missing {sorted(set(expected_keys) - set(type_keys))[:10]}")

# Check scholarship map
schol_match = re.search(r'const _scholarshipMap.*?= \{([\s\S]*?)\n\};', final)
schol_keys = sorted([int(m) for m in re.findall(r'^\s+(\d+):', schol_match.group(1), re.MULTILINE)])
print(f"_scholarshipMap keys ({len(schol_keys)}): ...{schol_keys[-5:]}")
if 193 in schol_keys:
    print("Scholarship entry 193 (Hosea 6:2) present ✓")
if 176 in schol_keys:
    print("Scholarship entry 176 (was 195) present ✓")
if 184 in schol_keys:
    print("Scholarship entry 184 (was 203) present ✓")
if 195 not in schol_keys and 203 not in schol_keys:
    print("Old keys 195, 203 removed ✓")

# Check for loop
if 'for (const p of prophecies)' in final:
    print("for loop present ✓")

# Check new entry
if '"raised-on-third-day"' in final:
    print("New Hosea 6:2 entry present ✓")

# Check removed entries aren't in makeLesson calls
removed_refs = ["Exodus 17:16", "Exodus 21:1", "Leviticus 26:13", "Judges 5:31",
    "Psalm 16:5", "Isaiah 10:27", "Isaiah 14:29", "Isaiah 30:18",
    "Isaiah 41:25", "Isaiah 51:12", "Isaiah 55:12", "Isaiah 62:10",
    "Jeremiah 5:19", "Ezekiel 11:19", "Daniel 8:13", "Amos 4:13",
    "Amos 5:18", "Zechariah 4:7", "Zechariah 8:12"]
problems = []
for ref in removed_refs:
    if f'"{ref}"' in final:
        # Check if it's in a makeLesson context
        context_idx = final.find(f'"{ref}"')
        surrounding = final[max(0,context_idx-100):context_idx+len(ref)+50]
        if 'makeLesson' in surrounding or 'prophecies.push' in surrounding:
            problems.append(ref)
if not problems:
    print("All 19 removed entries confirmed absent ✓")
else:
    print(f"WARNING: These refs still appear: {problems}")

# Check for unicode escape issues
if '\\u2019' in final or '\\u2014' in final:
    print("WARNING: Unicode escapes \\u2019/\\u2014 found — should use actual characters")
    cnt_2019 = final.count('\\u2019')
    cnt_2014 = final.count('\\u2014')
    print(f"  \\u2019 occurrences: {cnt_2019}, \\u2014 occurrences: {cnt_2014}")
else:
    print("No unicode escape issues ✓")

print(f"\nTotal file length: {len(final.splitlines())} lines")

Total makeLesson calls: 193
IDs are sequential 1-193 ✓
_prophecyTypeMap keys: 193 (should be 193)
Type map keys are sequential 1-193 ✓
_scholarshipMap keys (90): ...[97, 98, 176, 184, 193]
Scholarship entry 193 (Hosea 6:2) present ✓
Scholarship entry 176 (was 195) present ✓
Scholarship entry 184 (was 203) present ✓
Old keys 195, 203 removed ✓
for loop present ✓
New Hosea 6:2 entry present ✓
All 19 removed entries confirmed absent ✓
  \u2019 occurrences: 3, \u2014 occurrences: 5

Total file length: 1502 lines


In [4]:
# Fix unicode escapes - replace with actual characters
# \u2019 (right single quote) -> regular ASCII apostrophe '
# \u2014 (em dash) -> actual em dash character
with open(FILE, 'r') as f:
    final = f.read()

final = final.replace('\\u2019', '\u2019')  # This would make it a right quote, but existing file uses regular '
# Actually, let me check what character the existing entries use for apostrophes
# Look for "God's" in the file
gods_idx = final.find("God's")
if gods_idx != -1:
    apostrophe_char = final[gods_idx + 3]  # The apostrophe in God's
    print(f"Existing apostrophe char: U+{ord(apostrophe_char):04X} = {repr(apostrophe_char)}")

# Check for em dash usage
em_idx = final.find(' — ')
if em_idx != -1:
    dash_char = final[em_idx + 1]
    print(f"Existing em dash char: U+{ord(dash_char):04X} = {repr(dash_char)}")

# Replace literal \u2019 with regular apostrophe (ASCII ')
# Replace literal \u2014 with em dash (—)
with open(FILE, 'r', encoding='utf-8') as f:
    raw = f.read()

# The file contains literal \u2019 and \u2014 (as escape sequences)
raw = raw.replace('\\u2019', '\u2019')  # right single quote
raw = raw.replace('\\u2014', '\u2014')  # em dash

# But wait - we need regular ASCII apostrophe. Let me check what \u2019 produced
# \u2019 in Python = right single quotation mark '
# The existing entries use regular ASCII apostrophe '
# So let me use regular apostrophe instead
raw = raw.replace('\u2019', "'")

with open(FILE, 'w', encoding='utf-8') as f:
    f.write(raw)

# Verify
with open(FILE, 'r') as f:
    check = f.read()
if '\\u2019' in check or '\\u2014' in check:
    print("Still has unicode escapes!")
    print(f"\\u2019 count: {check.count(chr(0x5c) + 'u2019')}")
    print(f"\\u2014 count: {check.count(chr(0x5c) + 'u2014')}")
else:
    print("Unicode escapes fixed ✓")

# Double check the new entry looks right
hosea_idx = check.find('raised-on-third-day')
if hosea_idx != -1:
    snippet = check[hosea_idx:hosea_idx+200]
    print(f"Snippet: {snippet[:100]}")

Existing apostrophe char: U+0027 = "'"
Existing em dash char: U+2014 = '—'
Unicode escapes fixed ✓
Snippet: raised-on-third-day", "Raised on the Third Day", "Resurrection",
  "Hosea 6:2", "Luke 24:46",
  "Aft


In [5]:
# Final comprehensive verification
with open(FILE, 'r') as f:
    final = f.read()

# Re-run all checks
make_lesson_calls = re.findall(r'makeLesson\((\d+),', final)
ids = [int(m) for m in make_lesson_calls]
expected = list(range(1, 194))
assert ids == expected, f"ID mismatch!"
print(f"✓ {len(ids)} lessons, sequentially numbered 1-193")

# Count in main array vs pushed
main_array_count = final.count('makeLesson(') - final.count('prophecies.push(makeLesson(')
push_count = final.count('prophecies.push(makeLesson(')
print(f"✓ Main array: {main_array_count} lessons, pushed: {push_count} lessons")

# Verify the new entry content
hosea_start = final.find('prophecies.push(makeLesson(193,')
hosea_block = final[hosea_start:final.find('));', hosea_start)+3]
print(f"\n--- New entry (lesson 193) ---")
print(hosea_block[:300])
print("...")

# Verify type map entry for 193
type_match = re.search(r"193: '([^']+)'", final)
print(f"\nType map entry 193: {type_match.group(1)}")

# Verify scholarship map entry for 193
schol_193_idx = final.find("  193: { ...payne(78")
if schol_193_idx != -1:
    schol_line = final[schol_193_idx:final.find('\n', schol_193_idx)]
    print(f"Scholarship 193 present: {schol_line[:80]}...")
else:
    print("WARNING: Scholarship entry 193 not found!")

# Check for loop is intact
for_idx = final.find('for (const p of prophecies)')
for_block = final[for_idx:for_idx+200]
print(f"\nFor loop: {for_block[:150]}")

# Final summary
print(f"\n=== FINAL SUMMARY ===")
print(f"Total lessons: {len(ids)}")
print(f"File lines: {len(final.splitlines())}")
print(f"All checks passed ✓")

✓ 193 lessons, sequentially numbered 1-193
✓ Main array: 99 lessons, pushed: 95 lessons

--- New entry (lesson 193) ---
prophecies.push(makeLesson(193, "raised-on-third-day", "Raised on the Third Day", "Resurrection",
  "Hosea 6:2", "Luke 24:46",
  "After two days he will revive us; on the third day he will restore us, that we may live in his presence.",
  "He told them, \"This is what is written: The Messiah will su
...

Type map entry 193: Prophetic Pattern
Scholarship 193 present:   193: { ...payne(78, "Hos 6:2", "Revival after two days and raising on the thir...

For loop: for (const p of prophecies) {
  const s = _scholarshipMap[p.id];
  if (s) p.scholarship = { ...(p.scholarship ?? {}), ...s };
}



=== FINAL SUMMARY ===
Total lessons: 193
File lines: 1502
All checks passed ✓
